In [1]:
!which python3

/Users/tristantorchet/Desktop/Code/VSCode/SNN/.venv/bin/python3


In [2]:
import jax
import jax.numpy as jnp
import numpy as np
import data
import utils
from hyperparameters import SimArgs
import parameters
from sklearn.metrics import classification_report
from jax.example_libraries import optimizers

from utils import gr_than, train, inference
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
# check gpu with jax
print(jax.devices())

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
import sys
print(sys.executable)  # Shows the path to the Python interpreter
print(sys.version)     # Shows the Python version

In [ ]:
import wandb 

# get time hh:mm
import datetime
now = datetime.datetime.now()
h = now.hour
m = now.minute
d = now.day
mo = now.month


sweep_config = {
    'name': f'{d}_{mo}_{h}h{m}|PN21_R1_TrainHet_nh1_tm20_ts10_b_XavUni_RegS_',
    'method': 'grid',
    'metric': {
        'name': 'val_acc',
        'goal': 'maximize'
    },
    'parameters': {
        'seed':      {'values': [42, 43, 44]},    # random seed
        'tau_mem':   {'values': [20e-3]}, # membrane time constant
        'tau_syn':   {'values': [10e-3]}, # synaptic time constant
        'nb_epochs': {'values': [100]},    # number of epochs
        'lr':        {'values': [0.001]},  # learning rate
        'alpha_enabled': {'values': [True]}, # enable alpha synapses
        'bias_enable': {'values': [True]}, # add bias to each neuron in the hidden layer
        'output_reset': {'values': [False]}, # reset the output layer
        'save_weights': {'values': [False]}, # save the trained weights
        'llrr_factor': {'values': [0.1, 0.5, 1]}, # 
        'ugrr_factor': {'values': [0.005, 0.01, 0.05]}, #
        'llrr_thr': {'values': [0.02, 0.05, 0.1]}, #
        'ugrr_thr': {'values': [10, 15, 20]}, #

    },
}

In [ ]:
def main():
    wandb.init()
    alpha_enable = wandb.config.alpha_enabled
    output_reset = wandb.config.output_reset
    bias_enable = wandb.config.bias_enable
    args = SimArgs(
        layer_widths=[700, 128, 20],
        bias_enable=wandb.config.bias_enable,
        train_tau=True,
        seed=wandb.config.seed, 
        tau_mem=wandb.config.tau_mem,
        tau_syn=wandb.config.tau_syn,
        nb_epochs=wandb.config.nb_epochs, 
        lr=wandb.config.lr,
    )
    args.llrr_factor = wandb.config.llrr_factor
    args.ugrr_factor = wandb.config.ugrr_factor
    args.llrr_thr = wandb.config.llrr_thr
    args.ugrr_thr = wandb.config.ugrr_thr
    
    def lif_recurrent(state, input_spikes):
        ''' Vectorized Recurrent Leaky Integrate and Fire (LIF) neuron model
            :param state: ( (params, net_dyn), hp )
        '''
        
        print(f'{bias_enable=}, {alpha_enable=}, {output_reset=}, hey')
        # jax.debug.print("t++")
        tau_mem, v_th, beta_o, alpha_o, _, _ = state[1]
        params, net_dyn = state[0]
        net_dyn_new = []
        for layer_id, layer in enumerate(params):
            print(f'{layer_id=}') 
            i, v, z = net_dyn[layer_id]

            if layer_id != 0:
                print(f'zero')
                _, _, input_spikes = net_dyn[layer_id - 1]

            if layer_id == len(params) - 1:
                print(f'two')
                Win = layer[0]
                Wrec = 0 # no recurrent weights for the output layer, not efficient, just to be clear
                Wb = 0
                alpha = alpha_o
                beta = beta_o
            else: 
                if bias_enable:
                    Win, Wrec, Wb, alpha, beta = layer
                else: 
                    Win, Wrec, alpha, beta = layer
            if alpha_enable: 
                i = alpha * i + jnp.dot(Win, input_spikes)
            else:
                i = jnp.dot(Win, input_spikes)
            if layer_id < len(params) - 1: 
                print(f'rec')
                i += jnp.dot(Wrec, z) 
            if bias_enable: 
                i += Wb

            if layer_id == len(params) -1: 
                print(f'last')
                v = beta * v + i
                if output_reset: 
                    v -= z * v_th
            else: 
                v = beta * v + i - z * v_th
            #v = jnp.maximum(0, v)
            z = gr_than(v, v_th)
            net_dyn_new.append((i, v, z))
        
        
        return ((params, net_dyn_new), state[1]), net_dyn_new 

    utils.lif_recurrent = lif_recurrent    
    train_loader, val_loader, test_loader = data.get_data_loaders(args)

    key = jax.random.PRNGKey(args.seed)
    _, params, tau_boundaries = parameters.init_MLSNN(key, sim_params=args)
    params_init = params
    hp = (args.tau_mem, args.v_thr, 
        float(np.exp(-args.timestep/args.tau_mem)), 
        float(np.exp(-args.timestep/args.tau_syn)), 
        tau_boundaries, (args.llrr_factor, args.ugrr_factor, args.llrr_thr, args.ugrr_thr))    
    opt_init, opt_update, get_params = optimizers.adam(step_size=args.lr)
    opt_state = opt_init(params_init)

    loaders = data.get_data_loaders(args)
    get_params, opt_state, hist = train(params_init, hp, loaders, args)
    params_trained = get_params(opt_state)

    
    train_loss, train_acc, _ = inference(params_trained, hp, loaders[0])
    val_loss, val_acc, (val_labels, val_preds) = inference(params_trained, hp, loaders[1])
    print(f'{val_labels.shape=}, {val_preds.shape=}')
    test_loss, test_acc, _   = inference(params_trained, hp, loaders[2])
    
    
    # # if directory 'wandb_data' does not exist, create it
    # sim_path = f'wandb_data/pn21/r1/'
    # if not os.path.exists(sim_path):
    #     os.makedirs(sim_path)
    
    # # sim_id = f'pn21_nh{args.n_h}_tm{int(args.tau_mem*1e3)}_ts{int(args.tau_syn*1e3)}'
    # # if args.bias_enable:
    # #     sim_id += '_b'
    # # if args.pos_w:
    # #     sim_id += '_posW'
    # #     
    # # # create a directory for the current simulation
    # # sim_path += f'/{sim_id}'
    # # if not os.path.exists(sim_path):
    # #     os.makedirs(sim_path)
    
    # # read the csv file for train, val, test loss and accuracy
    # # check if results.csv exists
    # if not os.path.exists(f'{sim_path}/results.csv'):
    #     with open(f'{sim_path}/results.csv', 'w') as f:
    #         f.write('val_acc,test_acc,train_acc,val_loss,test_loss,train_loss,'
    #                 'n_h,nb_epochs,lr,bias_enable,pos_w,tau_mem,tau_syn,seed\n')
    # with open(f'{sim_path}/results.csv', 'a') as f:
    #     f.write(f'{val_acc.mean():.4f},{test_acc.mean():.4f},{train_acc.mean():.4f},'
    #             f'{val_loss.mean():.4f},{test_loss.mean():.4f},{train_loss.mean():.4f},'
    #             f'{args.n_h},{args.nb_epochs},{args.lr},{args.bias_enable},{args.pos_w},'
    #             f'{args.tau_mem},{args.tau_syn},{args.seed}\n')
    jax.clear_caches()
        



In [ ]:
os.environ["WANDB_NOTEBOOK_NAME"] = "/home/tristan/EDST/wandb_pn2021_Rn.ipynb"
sweep_id = wandb.sweep(sweep_config, project="SNN") 
wandb.login() 
wandb.agent(sweep_id, main)
